Для роботи алгоритму я використовую мінімальний набір стандартних інструментів Python. Завдяки оптимізації копіювання матриць, нам не потрібні важкі модулі на кшталт `copy`. Ми імпортуємо лише `random` для внесення непередбачуваності (випадкового вибору) у випадках, коли алгоритм знаходить кілька абсолютно рівноцінних ходів.

In [1]:
import random

Відповідно до формалізації за моделлю PEAS (Розділ 3.1.2 пояснювальної записки), середовище є двовимірною матрицею 8х8.

Клас `BoardState` відповідає за ініціалізацію початкової позиції та візуальне відображення дошки у консолі. Білі фішки позначаються маркером `w` (дамки `W`), чорні — `b` (дамки `B`), а порожні клітинки — `.`. Всі подальші математичні операції відбуватимуться безпосередньо з матрицею `self.board` для максимальної швидкодії.

In [2]:
class BoardState:
    def __init__(self):
        # Порожня дошка 8x8
        self.board = [['.' for _ in range(8)] for _ in range(8)]
        self._setup_initial_board()

    def _setup_initial_board(self):
        # Чорні (b) — верхні три ряди, Білі (w) — нижні три
        for row in range(3):
            for col in range(8):
                if (row + col) % 2 != 0:
                    self.board[row][col] = 'b'
        for row in range(5, 8):
            for col in range(8):
                if (row + col) % 2 != 0:
                    self.board[row][col] = 'w'

    def display(self, matrix=None):
        target = matrix if matrix else self.board
        print("  0 1 2 3 4 5 6 7")
        for i, row in enumerate(target):
            print(f"{i} {' '.join(row)}")
        print()

# Виведення проміжного результату
print("Початковий стан ігрового поля")
initial_board = BoardState()
initial_board.display()

Початковий стан ігрового поля
  0 1 2 3 4 5 6 7
0 . b . b . b . b
1 b . b . b . b .
2 . b . b . b . b
3 . . . . . . . .
4 . . . . . . . .
5 w . w . w . w .
6 . w . w . w . w
7 w . w . w . w .



Цей модуль є модулем основних правил та обмежень в рухах шашок детермінованого середовища. Клас `MoveGenerator` містить сувору логіку правил гри:
1. **Обов'язкове взяття:** Якщо агент може побити фішку суперника, звичайні ходи блокуються.
2. **Ланцюгові стрибки:** Метод `find_caps` використовує рекурсію для прорахунку багатокрокових взяттів. Якщо можна побити 2 або більше шашок за хід - алгоритм будує весь маршрут.
3. **Напрямок руху:** Шашки можуть бити у всі 4 сторони (включно зі стрибком назад), але звичайний хід роблять лише вперед.
*Оптимізація:* Замість повільного `deepcopy` (який був використаний в минулих версіях коду), для генерації майбутніх станів використовується швидке зрізове копіювання списків (`[row[:] for row in matrix]`), що запобігає перевантаженню пам'яті при глибокому зануренні алгоритму.

In [3]:
class MoveGenerator:
    def get_legal_moves(self, matrix, player):
        moves = []
        captures = []
        is_white = (player == 'w')
        forward = -1 if is_white else 1
        own = ['w', 'W'] if is_white else ['b', 'B']
        opp = ['b', 'B'] if is_white else ['w', 'W']

        def find_caps(r, c, path, caps, p_type, m_state):
            dirs = [(-1, -1), (-1, 1), (1, -1), (1, 1)]
            found = False
            for dr, dc in dirs:
                nr, nc = r + dr, c + dc
                jr, jc = r + 2*dr, c + 2*dc
                if 0 <= jr < 8 and 0 <= jc < 8:
                    if m_state[nr][nc] in opp and m_state[jr][jc] == '.' and (nr, nc) not in caps:
                        next_m = [row[:] for row in m_state]
                        next_m[nr][nc] = '.'
                        next_m[r][c] = '.'
                        next_m[jr][jc] = p_type
                        found = True
                        find_caps(jr, jc, path + [(jr, jc)], caps + [(nr, nc)], p_type, next_m)
            if not found and caps:
                captures.append((path, caps))

        for r in range(8):
            for c in range(8):
                if matrix[r][c] in own:
                    find_caps(r, c, [(r, c)], [], matrix[r][c], matrix)

        if captures: return captures

        for r in range(8):
            for c in range(8):
                p = matrix[r][c]
                if p in own:
                    dirs = [(-1,-1), (-1,1), (1,-1), (1,1)] if p.isupper() else [(forward, -1), (forward, 1)]
                    for dr, dc in dirs:
                        nr, nc = r + dr, c + dc
                        if 0 <= nr < 8 and 0 <= nc < 8 and matrix[nr][nc] == '.':
                            moves.append(([(r, c), (nr, nc)], []))
        return moves

    def apply_move(self, matrix, move):
        new_m = [row[:] for row in matrix]
        path, caps = move
        start, end = path[0], path[-1]
        p = new_m[start[0]][start[1]]
        new_m[start[0]][start[1]] = '.'
        new_m[end[0]][end[1]] = p
        for cr, cc in caps: new_m[cr][cc] = '.'
        if p == 'w' and end[0] == 0: new_m[end[0]][end[1]] = 'W'
        elif p == 'b' and end[0] == 7: new_m[end[0]][end[1]] = 'B'
        return new_m

# Виведення проміжного результату
print("Демонстрація логіки ходів")
gen = MoveGenerator()
test_matrix = BoardState().board
moves_demo = gen.get_legal_moves(test_matrix, 'w')
print(f"Доступно ходів для білих: {len(moves_demo)}")
print(f"Приклад першого ходу: {moves_demo[0][0][0]} -> {moves_demo[0][0][-1]}")
test_matrix = gen.apply_move(test_matrix, moves_demo[0])
initial_board.display(test_matrix)

Демонстрація логіки ходів
Доступно ходів для білих: 7
Приклад першого ходу: (5, 0) -> (4, 1)
  0 1 2 3 4 5 6 7
0 . b . b . b . b
1 b . b . b . b .
2 . b . b . b . b
3 . . . . . . . .
4 . w . . . . . .
5 . . w . w . w .
6 . w . w . w . w
7 w . w . w . w .



Згідно з Розділом 4.1.2 пояснювальної записки, тут реалізовано 3 математичні моделі оцінки позиції:
- **$H_1$ (Матеріальна):** Оцінює різницю у кількості фішок (1 бал за шашку, 3 бали за дамку).
- **$H_2$ (Позиційна):** Базується на $H_1$, але додає просторовий контекст — бонуси за контроль центру (колонки 2-5) та просування до зони перетворення в дамки.
- **$H_3$ (Тактична):** Найскладніша евристика. Враховує матеріал, позицію, а також *мобільність* (кількість доступних легальних ходів) та *безпеку* (захищеність фішок на краях дошки).

In [4]:
class Heuristics:
    def h1_material(self, matrix, player):
        s = 0
        is_w = (player == 'w')
        own, opp = (['w', 'W'], ['b', 'B']) if is_w else (['b', 'B'], ['w', 'W'])
        for r in range(8):
            for c in range(8):
                p = matrix[r][c]
                if p in own: s += (3 if p.isupper() else 1)
                elif p in opp: s -= (3 if p.isupper() else 1)
        return s

    def h2_positional(self, matrix, player):
        s = self.h1_material(matrix, player)
        own = ['w', 'W'] if player == 'w' else ['b', 'B']
        for r in range(8):
            for c in range(8):
                if matrix[r][c] in own:
                    if 2 <= c <= 5: s += 0.2
                    if (player == 'w' and r <= 2) or (player == 'b' and r >= 5): s += 0.3
        return s

    def h3_tactical(self, matrix, player):
        s = self.h2_positional(matrix, player)
        s += len(MoveGenerator().get_legal_moves(matrix, player)) * 0.1
        own = ['w', 'W'] if player == 'w' else ['b', 'B']
        for r in range(8):
            for c in range(8):
                if matrix[r][c] in own and (c == 0 or c == 7): s += 0.2
        return s

Клас `AlphaBetaAgent` реалізує алгоритм змагального пошуку з Альфа-Бета відсіканням (Alpha-Beta Pruning).
Агент генерує дерево можливих ходів на задану глибину (`depth`), почергово симулюючи ходи за себе (MAX) та за суперника (MIN). Гілки дерева, які гарантовано гірші за вже знайдені альтернативи, "відсікаються", що значно економить обчислювальний час. Досягнувши ліміту глибини, агент викликає інжектовану евристику ($H_1$, $H_2$ або $H_3$) для оцінки фінального стану.

In [5]:
class AlphaBetaAgent:
    def __init__(self, h_name, depth=3, player='w'):
        self.h = Heuristics()
        self.g = MoveGenerator()
        self.h_func = getattr(self.h, h_name)
        self.depth, self.player = depth, player

    def get_best_move(self, matrix):
        moves = self.g.get_legal_moves(matrix, self.player)
        if not moves: return None, -1000

        # ВИПРАВЛЕННЯ: Перемішуємо ходи для уникнення однакових партій
        random.shuffle(moves)

        best, max_v = moves[0], float('-inf')
        a, b = float('-inf'), float('inf')

        for m in moves:
            v = self._ab(self.g.apply_move(matrix, m), self.depth-1, a, b, False)
            if v > max_v:
                max_v, best = v, m
            a = max(a, v)
        return best, max_v

    def _ab(self, mtx, d, a, b, maxi):
        if d == 0: return self.h_func(mtx, self.player)
        p = self.player if maxi else ('b' if self.player == 'w' else 'w')
        moves = self.g.get_legal_moves(mtx, p)
        if not moves: return float('-inf') if maxi else float('inf')
        v = float('-inf') if maxi else float('inf')
        for m in moves:
            res = self._ab(self.g.apply_move(mtx, m), d-1, a, b, not maxi)
            v = max(v, res) if maxi else min(v, res)
            if maxi: a = max(a, v)
            else: b = min(b, v)
            if b <= a: break
        return v
# Виведення проміжного результату
print("Порівняння оцінок евристик для початкової позиції")
start_mtx = BoardState().board
print(f"{'Евристика':<20} | {'Оцінка ШІ (d=5)':<15} | {'Найкращий хід'}")
print("-" * 55)
for h in ['h1_material', 'h2_positional', 'h3_tactical']:
    agent = AlphaBetaAgent(h, depth=5)
    move, score = agent.get_best_move(start_mtx)
    print(f"{h:<20} | {score:<15.2f} | {move[0][0]}->{move[0][-1]}")

Порівняння оцінок евристик для початкової позиції
Евристика            | Оцінка ШІ (d=5) | Найкращий хід
-------------------------------------------------------
h1_material          | 0.00            | (5, 2)->(4, 1)
h2_positional        | 1.40            | (5, 0)->(4, 1)
h3_tactical          | 2.70            | (5, 2)->(4, 1)


Для збору статистично значущих даних ми проводимо автоматизований круговий турнір. Агенти з різними евристиками грають один проти одного серію партій.
Щоб експеримент був абсолютно чистим і об'єктивним:
* Агенти по черзі міняються кольорами (нівелюється перевага першого ходу).
* Параметри турніру (кількість партій, максимальна кількість ходів для уникнення зациклень та глибина занурення) винесені у зручний блок налаштувань.
Результати виводяться у вигляді форматованої таблиці, що дозволяє легко порівняти ефективність кожної евристики.

Задано три основні параметри які впливають на результат нашого турніру, це:
- `GAMES_PER_MATCHUP` - Кількість партій між двома агентами
- `MAX_MOVES_PER_GAME` - Максимальна кількість півходів у одній партії (захист від зациклення)
- `SEARCH_DEPTH` - Глибина прорахунку алгоритму

In [17]:
def run_simulation(h1_name, h2_name, num_games, depth, max_moves):
    wins1, wins2, draws = 0, 0, 0
    mg = MoveGenerator()

    for i in range(num_games):
        matrix = BoardState().board
        # Для чесності змінюємо колір першого агента кожну гру
        if i % 2 == 0:
            agent_w = AlphaBetaAgent(h1_name, depth, 'w')
            agent_b = AlphaBetaAgent(h2_name, depth, 'b')
            p1_color = 'w'
        else:
            agent_w = AlphaBetaAgent(h2_name, depth, 'w')
            agent_b = AlphaBetaAgent(h1_name, depth, 'b')
            p1_color = 'b'

        turn = 'w'
        for _ in range(max_moves): # Використовуємо заданий ліміт ходів на партію
            agent = agent_w if turn == 'w' else agent_b
            move, _ = agent.get_best_move(matrix)

            if not move: # Якщо ходів немає - гравець програв
                winner_color = 'b' if turn == 'w' else 'w'
                if winner_color == p1_color:
                    wins1 += 1
                else:
                    wins2 += 1
                break

            matrix = mg.apply_move(matrix, move)
            turn = 'b' if turn == 'w' else 'w'
        else:
            draws += 1

    return wins1, wins2, draws


# НАЛАШТУВАННЯ ТУРНІРУ

GAMES_PER_MATCHUP = 40
MAX_MOVES_PER_GAME = 150
SEARCH_DEPTH = 5


matchups = [
    ("H1 (Матеріал)", "h1_material", "H2 (Позиція)", "h2_positional"),
    ("H2 (Позиція)", "h2_positional", "H3 (Тактика)", "h3_tactical"),
    ("H1 (Матеріал)", "h1_material", "H3 (Тактика)", "h3_tactical")
]

print(f"Результати турніру")
print(f"Глибина пошуку: {SEARCH_DEPTH}")
print(f"Ліміт півходів на партію: {MAX_MOVES_PER_GAME}")
print(f"Партій на кожен матч: {GAMES_PER_MATCHUP}\n")

# Оновлений заголовок таблиці з новою колонкою
header = f"{'Матч':<35} | {'Пер. 1':<8} | {'Пер. 2':<8} | {'Нічиї':<6} | {'Лідер (% перемог)'}"
print(header)
print("-" * len(header))

for name1, func1, name2, func2 in matchups:
    w1, w2, d = run_simulation(
        func1,
        func2,
        num_games=GAMES_PER_MATCHUP,
        depth=SEARCH_DEPTH,
        max_moves=MAX_MOVES_PER_GAME
    )

    # Визначення лідера та обчислення відсотка перемог
    if w1 > w2:
        win_percent = (w1 / GAMES_PER_MATCHUP) * 100
        leader_info = f"{name1} ({win_percent:.1f}%)"
    elif w2 > w1:
        win_percent = (w2 / GAMES_PER_MATCHUP) * 100
        leader_info = f"{name2} ({win_percent:.1f}%)"
    else:
        leader_info = "Однаковий відсоток перемог"

    print(f"{name1 + ' vs ' + name2:<35} | {w1:<8} | {w2:<8} | {d:<6} | {leader_info}")

Результати турніру
Глибина пошуку: 5
Ліміт півходів на партію: 150
Партій на кожен матч: 40

Матч                                | Пер. 1   | Пер. 2   | Нічиї  | Лідер (% перемог)
--------------------------------------------------------------------------------------
H1 (Матеріал) vs H2 (Позиція)       | 4        | 16       | 20     | H2 (Позиція) (40.0%)
H2 (Позиція) vs H3 (Тактика)        | 7        | 15       | 18     | H3 (Тактика) (37.5%)
H1 (Матеріал) vs H3 (Тактика)       | 6        | 11       | 23     | H3 (Тактика) (27.5%)
